# Orquestación Automática de Respuesta a Incidentes de Seguridad

## Introducción a la Automatización de Respuesta

La IA puede automatizar decisiones críticas en el ciclo de respuesta a incidentes:
**detección → análisis → clasificación → respuesta → logging**

Este cuaderno implementa:
- **Triaje automático**: clasificar incidentes por severidad (SVM)
- **Orquestación de respuestas**: Motor de decisión para ejecutar acciones automatizadas
- **Logging y análisis post-incidente** para auditoría y mejora continua


### Generación de Datos Simulados de Eventos de Seguridad

Se crea un dataset sintético de incidentes con características clave:
severidad, tipo de ataque, protocolos afectados, señales IDS/IPS.


In [ ]:
import numpy as np
import pandas as pd
import os

np.random.seed(42)
os.makedirs('data', exist_ok=True)

n = 2000

# Severidad: 0=bajo, 1=medio, 2=alto, 3=crítico
severities = np.random.choice([0, 1, 2, 3], n, p=[0.40, 0.30, 0.20, 0.10])

records = []
for sev in severities:
    if sev == 0:    # Bajo
        row = {
            'num_hosts_afectados':   np.random.randint(1, 2),
            'tipo_evento_cod':       np.random.choice([1, 2]),
            'bytes_exfiltrados':     np.random.randint(0, 500),
            'duracion_seg':          np.random.randint(1, 30),
            'privilegios_elevados':  0,
        }
    elif sev == 1:  # Medio
        row = {
            'num_hosts_afectados':   np.random.randint(1, 5),
            'tipo_evento_cod':       np.random.choice([2, 3, 4]),
            'bytes_exfiltrados':     np.random.randint(500, 10000),
            'duracion_seg':          np.random.randint(15, 120),
            'privilegios_elevados':  np.random.choice([0, 1], p=[0.7, 0.3]),
        }
    elif sev == 2:  # Alto
        row = {
            'num_hosts_afectados':   np.random.randint(3, 15),
            'tipo_evento_cod':       np.random.choice([4, 5, 6]),
            'bytes_exfiltrados':     np.random.randint(10000, 500000),
            'duracion_seg':          np.random.randint(60, 600),
            'privilegios_elevados':  np.random.choice([0, 1], p=[0.3, 0.7]),
        }
    else:           # Crítico
        row = {
            'num_hosts_afectados':   np.random.randint(10, 50),
            'tipo_evento_cod':       np.random.choice([5, 6, 7]),
            'bytes_exfiltrados':     np.random.randint(500000, 5000000),
            'duracion_seg':          np.random.randint(300, 3600),
            'privilegios_elevados':  1,
        }
    row['severity'] = sev
    records.append(row)

df_inc = pd.DataFrame(records).sample(frac=1, random_state=42).reset_index(drop=True)
df_inc.to_csv('data/incident_data.csv', index=False)

print(f"[OK] Dataset simulado guardado: data/incident_data.csv ({len(df_inc)} registros)")
print(f"     Distribución de severidad:")
print(f"     {df_inc['severity'].value_counts().sort_index().to_dict()}")
print(f"     0=Bajo, 1=Medio, 2=Alto, 3=Crítico")
df_inc.head()

---

## Preparación de Características de Incidentes


In [ ]:
# Listing 5.1: Clasificación de incidentes por severidad con SVM

import pandas as pd
import numpy as np
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# 1. Cargar datos de incidentes
# Columnas esperadas: num_hosts_afectados, tipo_evento_cod,
# bytes_exfiltrados, duracion_seg, privilegios_elevados, etc.
# Etiqueta: 0=bajo, 1=medio, 2=alto, 3=crítico
incidents = pd.read_csv('data/incident_data.csv').dropna()
X = incidents.drop('severity', axis=1)
y = incidents['severity']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# 2. Pipeline: escalado + SVM
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='rbf', C=1.0, gamma='scale',
                probability=True, random_state=42))
])
pipeline.fit(X_train, y_train)

# 3. Evaluación
y_pred = pipeline.predict(X_test)
etiquetas = ['Bajo', 'Medio', 'Alto', 'Crítico']
print("=== Reporte de clasificación ===")
print(classification_report(y_test, y_pred, target_names=etiquetas))

In [ ]:
# 4. Probabilidades para un nuevo incidente
nuevo_incidente = pd.DataFrame([{
    'num_hosts_afectados': 3,
    'tipo_evento_cod':     5,
    'bytes_exfiltrados':   102400,
    'duracion_seg':        120,
    'privilegios_elevados': 1
}])
proba = pipeline.predict_proba(nuevo_incidente)[0]
pred = pipeline.predict(nuevo_incidente)[0]
print(f"Nuevo incidente → Severidad predicha: {etiquetas[int(pred)]}")
for i, p in enumerate(proba):
    print(f"  P({etiquetas[i]}) = {p:.3f}")

### Análisis de Resultados: Matriz de Confusión


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# Matriz de confusión
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=etiquetas)
disp.plot(cmap='Reds')
plt.title('Matriz de Confusión - Triaje SVM')
plt.tight_layout()
plt.savefig('data/confusion_matrix_svm_incidents.png', dpi=150)
plt.show()

---

## Motor de Orquestación: Respuestas Automáticas

Una vez clasificado el incidente por severidad, el motor de orquestación ejecuta
respuestas automáticas como bloqueo, aislamiento, notificaciones, etc.

> **Nota:** Las integraciones en este entorno son simuladas para desarrollo seguro.
> En producción, conectar con APIs de firewall, SOAR, SIEM reales.


In [ ]:
# Listing 5.2: Motor de respuesta automática ante incidentes

import requests
import logging
from dataclasses import dataclass
from enum import IntEnum

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)-8s %(message)s'
)

# === MODO SIMULADO ===
# Cambiar a False en producción y configurar BASE_URL / TOKEN reales
SIMULATE = True
BASE_URL = "https://security-platform/api"
AUTH_TOKEN = "Bearer <TOKEN>"


class Severidad(IntEnum):
    BAJO = 0
    MEDIO = 1
    ALTO = 2
    CRITICO = 3


@dataclass
class Incidente:
    id_sistema: str
    ip_origen: str
    severidad: Severidad
    descripcion: str


def aislar_sistema(sistema_id: str) -> bool:
    """Envía orden de aislamiento de red al sistema comprometido."""
    if SIMULATE:
        logging.info(f"[SIMULADO] Sistema {sistema_id} aislado correctamente.")
        return True
    try:
        resp = requests.post(
            f"{BASE_URL}/isolate/{sistema_id}",
            timeout=10,
            headers={"Authorization": AUTH_TOKEN}
        )
        resp.raise_for_status()
        logging.info(f"Sistema {sistema_id} aislado correctamente.")
        return True
    except requests.RequestException as e:
        logging.error(f"Error al aislar sistema {sistema_id}: {e}")
        return False


def bloquear_ip(ip: str) -> bool:
    """Agrega la IP al bloqueo en el firewall perimetral."""
    if SIMULATE:
        logging.info(f"[SIMULADO] IP {ip} bloqueada en firewall.")
        return True
    try:
        resp = requests.post(
            f"{BASE_URL}/firewall/block",
            json={"ip": ip},
            timeout=10,
        )
        resp.raise_for_status()
        logging.info(f"IP {ip} bloqueada en firewall.")
        return True
    except requests.RequestException as e:
        logging.error(f"Error al bloquear IP {ip}: {e}")
        return False


def notificar_equipo(incidente: Incidente) -> None:
    """Envía alerta al equipo de respuesta (webhook)."""
    payload = {
        "text": (
            f"*ALERTA SEGURIDAD* Severidad: {incidente.severidad.name}\n"
            f"Sistema: {incidente.id_sistema}\n"
            f"IP: {incidente.ip_origen}\n"
            f"Descripción: {incidente.descripcion}"
        )
    }
    if SIMULATE:
        logging.info(f"[SIMULADO] Notificación enviada al equipo.")
        return
    try:
        requests.post(f"{BASE_URL}/notify", json=payload, timeout=5)
        logging.info("Notificación enviada al equipo.")
    except requests.RequestException as e:
        logging.warning(f"Fallo en notificación: {e}")


def orquestar_respuesta(incidente: Incidente) -> None:
    """
    Selecciona y ejecuta acciones según la severidad del incidente.
    """
    logging.info(
        f"[Incidente] Sistema={incidente.id_sistema} "
        f"Severidad={incidente.severidad.name}"
    )

    if incidente.severidad == Severidad.BAJO:
        logging.info("Acción: registrar y monitorear.")

    elif incidente.severidad == Severidad.MEDIO:
        notificar_equipo(incidente)

    elif incidente.severidad == Severidad.ALTO:
        bloquear_ip(incidente.ip_origen)
        notificar_equipo(incidente)

    elif incidente.severidad == Severidad.CRITICO:
        aislar_sistema(incidente.id_sistema)
        bloquear_ip(incidente.ip_origen)
        notificar_equipo(incidente)


print("[OK] Motor de orquestación de respuestas definido.")
print("     Funciones: aislar_sistema(), bloquear_ip(), notificar_equipo(), orquestar_respuesta()")

### Demostración del Motor con Múltiples Escenarios


In [ ]:
# Ejemplo de uso con diferentes niveles de severidad

# --- Incidente BAJO ---
inc_bajo = Incidente(
    id_sistema="WS-USER-101",
    ip_origen="10.0.1.45",
    severidad=Severidad.BAJO,
    descripcion="Intento de login fallido aislado."
)
orquestar_respuesta(inc_bajo)

print("---")

# --- Incidente MEDIO ---
inc_medio = Incidente(
    id_sistema="WS-DEV-205",
    ip_origen="10.0.2.88",
    severidad=Severidad.MEDIO,
    descripcion="Escaneo de puertos detectado desde host interno."
)
orquestar_respuesta(inc_medio)

print("---")

# --- Incidente ALTO ---
inc_alto = Incidente(
    id_sistema="DB-PROD-003",
    ip_origen="192.168.10.77",
    severidad=Severidad.ALTO,
    descripcion="Acceso no autorizado a base de datos de producción."
)
orquestar_respuesta(inc_alto)

print("---")

# --- Incidente CRÍTICO ---
inc_critico = Incidente(
    id_sistema="SRV-PROD-042",
    ip_origen="192.168.10.55",
    severidad=Severidad.CRITICO,
    descripcion="Exfiltración masiva de datos detectada."
)
orquestar_respuesta(inc_critico)

### Pipeline Completo: Triaje Automático + Orquestación


In [ ]:
def triaje_automatico(datos_incidente: dict, id_sistema: str, ip_origen: str):
    """
    Flujo completo: recibe datos crudos del incidente,
    predice la severidad con el modelo SVM y ejecuta la respuesta.
    """
    # 1. Preparar datos para el modelo
    df_input = pd.DataFrame([datos_incidente])
    
    # 2. Predecir severidad
    severidad_pred = int(pipeline.predict(df_input)[0])
    proba = pipeline.predict_proba(df_input)[0]
    
    etiquetas_sev = ['Bajo', 'Medio', 'Alto', 'Crítico']
    print(f"\n{'='*50}")
    print(f"TRIAJE AUTOMÁTICO")
    print(f"{'='*50}")
    print(f"Severidad predicha: {etiquetas_sev[severidad_pred]}")
    for i, p in enumerate(proba):
        print(f"  P({etiquetas_sev[i]}) = {p:.3f}")
    
    # 3. Crear objeto incidente y orquestar
    inc = Incidente(
        id_sistema=id_sistema,
        ip_origen=ip_origen,
        severidad=Severidad(severidad_pred),
        descripcion=f"Incidente auto-clasificado como {etiquetas_sev[severidad_pred]}."
    )
    orquestar_respuesta(inc)


# Ejemplo: incidente en tiempo real
triaje_automatico(
    datos_incidente={
        'num_hosts_afectados': 12,
        'tipo_evento_cod':     6,
        'bytes_exfiltrados':   850000,
        'duracion_seg':        540,
        'privilegios_elevados': 1
    },
    id_sistema="SRV-WEB-007",
    ip_origen="172.16.5.200"
)

---

## Almacenamiento de Componentes del Sistema


In [ ]:
import joblib
import os

os.makedirs('models', exist_ok=True)

# Guardar pipeline SVM
joblib.dump(pipeline, 'models/svm_incident_triage.pkl')
print("[OK] Pipeline SVM guardado en models/svm_incident_triage.pkl")

print("\n=== Resumen de artefactos generados ===")
print("Datos:   data/incident_data.csv")
print("Modelo:  models/svm_incident_triage.pkl")
print("Gráfico: data/confusion_matrix_svm_incidents.png")